# Generate and save supplementary tables
Changelog:

20260916
- propagate survival data across samples from the same patient. survival annotations for additional 34 cases.
- Aggregate multiracial in S.T.s
- add tumor fraction
- prioritize extant metadata for deduplicating biosamples

20260928
- removed cases for which consent was withdrawn
- Remove BS_HJ7HYZ7N, probable normal mis-annotated as tumor.
- Remove 2 probable cases of 1 patient assigned 2 PT ids.
- CAVATICA metadata mislabels ages at sample collection as age at diagnosis. Prefer openPBTA histologies where available.
- Likewise, prefer openPBTA tumor histories over CAVATICA
- Fix Excel autocorrecting dates
- New age_at_collection for CBTN samples
- Add new tests
- New excel export function

20261007
- remove X samples identified as cell line in the latest OpenPanCan release. As a result,
  - 1 PT excluded, CPT, intrachromosomal
  - 32 BS excluded (true cell lines); 2 added (previously incorrectly ID'd as cell line) 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
sys.path.append('../../src')
from data_imports import *
import tests
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import datetime
import numbers
import shutil
from openpyxl import load_workbook

## Regenerate tables, run tests

In [ ]:
biosamples = generate_biosample_table()
print(f"Number of unique biosamples:{len(biosamples)}")
print(f"Number of unique tumors: {biosamples.in_unique_tumor_set.sum()}")
print(f"Number of unique patients:{biosamples.in_unique_patient_set.sum()}")
biosamples.head()

In [ ]:
patients = generate_patient_table()
patients.head()

In [ ]:
amplicons = generate_amplicon_table(biosamples)
amplicons.head()

In [ ]:
genes = generate_gene_table(biosamples)
genes.head()

In [ ]:
# Run tests
tests.run_all_tests(patients=patients,biosamples=biosamples,amplicons=amplicons)

## Examine any differences between latest and previous S.T. versions

In [ ]:
patients_old = import_patients()
biosamples_old = import_biosamples()
amplicons_old = import_amplicons()
genes_old = import_genes()

In [ ]:
def _keyed(df, key=None):
    '''
    Index df by key (a column name or list of column names; default: the existing index), as strings.
    Raises if the key is not unique, since rows could not be matched unambiguously.
    '''
    df = df.set_index(key) if key is not None else df.copy()
    if isinstance(df.index, pd.MultiIndex):
        df.index = pd.MultiIndex.from_arrays([df.index.get_level_values(i).astype(str) for i in range(df.index.nlevels)], names=df.index.names)
    else:
        df.index = df.index.astype(str)
    dups = df.index[df.index.duplicated()].unique()
    if len(dups) > 0:
        raise ValueError(f"Key is not unique: {len(dups)} duplicated, e.g. {list(dups[:5])}")
    return df


def compare_columns(new, old):
    '''
    Print and return the columns added to and removed from old.
    '''
    added = [c for c in new.columns if c not in old.columns]
    removed = [c for c in old.columns if c not in new.columns]
    print(f"Columns added: {added if added else 'none'}")
    print(f"Columns removed: {removed if removed else 'none'}")
    return added, removed


def compare_rows(new, old, key=None):
    '''
    Print counts and display the full rows added to and removed from old, matched on key (default: index).
    Returns (added rows from new, removed rows from old).
    '''
    new, old = _keyed(new, key), _keyed(old, key)
    added = new[~new.index.isin(old.index)]
    removed = old[~old.index.isin(new.index)]
    print(f"Rows added: {len(added)}")
    if len(added) > 0:
        display(added)
    print(f"Rows removed: {len(removed)}")
    if len(removed) > 0:
        display(removed)
    return added, removed


def _normalize(s):
    '''
    Normalize a column for comparison: missing values (NaN, None, empty strings) become NaN, everything else becomes stripped text.
    '''
    NA_VALUES = ["unknown"] # treated as missing as in import_genes when AC writes 'unknown' gene_cn
    text = s.astype(str).str.strip()
    return text.mask(s.isna() | (text == "") | text.isin(NA_VALUES))


def _differs(a, b, rtol=1e-9):
    '''
    Boolean Series: True where a and b differ, ignoring type changes.
    Cells that are numbers in both are compared as numbers (within rtol); all others as text.
    '''
    a_text, b_text = _normalize(a), _normalize(b)
    a_num, b_num = pd.to_numeric(a_text, errors="coerce"), pd.to_numeric(b_text, errors="coerce")
    same_number = np.isclose(a_num, b_num, rtol=rtol, atol=0)
    same_text = (a_text == b_text) | (a_text.isna() & b_text.isna())
    return ~(same_number | same_text)


def compare_values(new, old, key=None):
    '''
    Display changed values in the rows and columns shared by new and old (via DataFrame.compare), ignoring type changes.
    Returns the DataFrame.compare output.
    '''
    new, old = _keyed(new, key), _keyed(old, key)
    rows = new.index[new.index.isin(old.index)]
    cols = [c for c in new.columns if c in old.columns]
    new, old = new.loc[rows, cols], old.loc[rows, cols]
    diff = pd.DataFrame({c: _differs(new[c], old[c]) for c in cols})
    # Blank out unchanged cells in both tables, so DataFrame.compare shows only real differences, with their original values.
    changes = new.astype(object).where(diff).compare(old.astype(object).where(diff), result_names=("new", "old"))
    n_cols = diff.any().sum()
    print(f"Values changed: {int(diff.values.sum())} in {int(diff.any(axis=1).sum())} rows, across {n_cols} columns"
          + (f": {list(diff.columns[diff.any()])}" if n_cols else ""))
    if len(changes) > 0:
        display(changes)
    return changes


def compare_tables(new, old, key=None):
    '''
    Compare a new version of a table against an old one: columns added/removed, rows added/removed, and changed values.
    key: column name(s) identifying a row (default: the index).
    Returns a dict of the results.
    '''
    columns_added, columns_removed = compare_columns(new, old)
    rows_added, rows_removed = compare_rows(new, old, key)
    changes = compare_values(new, old, key)
    return {"columns_added": columns_added, "columns_removed": columns_removed,
            "rows_added": rows_added, "rows_removed": rows_removed, "changes": changes}

In [ ]:
_ = compare_tables(patients,patients_old)

In [ ]:
_ = compare_tables(biosamples, biosamples_old)

In [ ]:
amplicon_key = ['sample_name','amplicon_number']
_ = compare_tables(amplicons, amplicons_old, key=amplicon_key)

In [ ]:
genes_key = ['sample_name','amplicon_number','feature','gene']
_ = compare_tables(genes, genes_old, key=genes_key)

In [ ]:
patients.race.unique()

In [ ]:
patients.ethnicity.unique()

In [ ]:
patients.cohort.unique()

## save new Suppl. Tables

In [ ]:
def archive_workbook(path, archive_dir=None):
    '''
    Copy the workbook to <archive_dir>/<name>_<YYYYMMDD-HHMMSS>.xlsx (default archive_dir: an "archive" folder next to it).
    Returns the path of the archived copy.
    '''
    path = pathlib.Path(path)
    archive_dir = pathlib.Path(archive_dir) if archive_dir else path.parent / "archive"
    archive_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    dest = archive_dir / f"{path.stem}_{stamp}{path.suffix}"
    shutil.copy2(path, dest)
    return dest

def _cell_value(v):
    '''
    Convert a DataFrame value to an openpyxl cell value. Returns (value, is_text).
    Missing values become empty cells; booleans and numbers keep their type; everything else is written as text.
    '''
    if v is None or v is pd.NA or v is pd.NaT or (isinstance(v, float) and np.isnan(v)):
        return None, False
    if isinstance(v, (bool, np.bool_)):
        return bool(v), False
    if isinstance(v, numbers.Integral):
        return int(v), False
    if isinstance(v, numbers.Real):
        return float(v), False
    return str(v), True

def write_supplementary_tables(tables, path=SUPPLEMENTARY_TABLES_PATH, archive=True, archive_dir=None):
    '''
    Replace the contents of existing sheets in the Supplementary Tables workbook; all other sheets are left untouched.
    tables: {sheet name: DataFrame}. A named index (eg. biosample_id) is written as the first column.
    Strings are written as text cells ('@' format), so Excel never reinterprets them (eg. '7316-3' as a date).
    archive: copy the existing workbook to a timestamped file first.
    '''
    path = pathlib.Path(path)
    lock = path.with_name("~$" + path.name)
    if lock.exists():
        raise RuntimeError(f"{path.name} appears to be open in Excel ({lock.name} exists); close it first.")
    wb = load_workbook(path)
    missing = [sheet for sheet in tables if sheet not in wb.sheetnames]
    if missing:
        raise KeyError(f"Sheets not found in {path.name}: {missing}")
    if archive:
        print(f"Archived previous version to {archive_workbook(path, archive_dir)}")

    for sheet, df in tables.items():
        df = df.reset_index() if df.index.name else df
        ws = wb[sheet]
        ws.delete_rows(1, ws.max_row)
        for c, name in enumerate(df.columns, start=1):
            cell = ws.cell(row=1, column=c, value=str(name))
            cell.number_format = "@"
        for r, row in enumerate(df.itertuples(index=False), start=2):
            for c, v in enumerate(row, start=1):
                value, is_text = _cell_value(v)
                cell = ws.cell(row=r, column=c, value=value)
                if is_text:
                    cell.number_format = "@"
        print(f"Wrote {len(df)} rows x {len(df.columns)} columns to '{sheet}'")
    wb.save(path)

In [ ]:
write_supplementary_tables({
    "1. Patients": patients,
    "2. Biosamples": biosamples,
    "3. Amplicons": amplicons,
    "5. Gene amplifications": genes,
}, archive=True, archive_dir='out/archive')